In [1]:
import torch
import os
import yaml
import pandas as pd
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
SPLIT_ROOT = r"C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS"

TRAIN_IMG = os.path.join(SPLIT_ROOT, "Data Train", "images")
VAL_IMG   = os.path.join(SPLIT_ROOT, "Data Validation", "images")
TEST_IMG  = os.path.join(SPLIT_ROOT, "Data Test", "images")

CLASS_NAMES = [
    "atrophic_scar", "comedo", "hypertrophic_scar", "melasma",
    "nevus", "nodule", "other", "papule", "pustule",
]

for p in [TRAIN_IMG, VAL_IMG, TEST_IMG]:
    assert os.path.isdir(p), f"Directory not found: {p}"
    n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"{p}  ->  {n} images")

DATA_YAML = os.path.join(SPLIT_ROOT, "baseline_yolov8n.yaml")
yaml_content = {
    "path": SPLIT_ROOT,
    "train": os.path.join("Data Train", "images"),
    "val":   os.path.join("Data Validation", "images"),
    "test":  os.path.join("Data Test", "images"),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
with open(DATA_YAML, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"\ndata.yaml saved to: {DATA_YAML}")
print(yaml.dump(yaml_content, default_flow_style=False))

C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\Data Train\images  ->  170 images
C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\Data Validation\images  ->  56 images
C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\Data Test\images  ->  50 images

data.yaml saved to: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\baseline_yolov8n.yaml
names:
- atrophic_scar
- comedo
- hypertrophic_scar
- melasma
- nevus
- nodule
- other
- papule
- pustule
nc: 9
path: C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS
test: Data Test\images
train: Data Train\images
val: Data Validation\images



In [3]:
PROJECT_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "Baseline Model Yolov8n", "runs")

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=300,
    patience=50,
    project=PROJECT_DIR,
    name="baseline_yolov8n",
    exist_ok=True,
    verbose=True,
)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.40  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\baseline_yolov8n.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

In [6]:
from ultralytics import settings

# View all current settings
print(settings)

JSONDict("C:\Users\delim\AppData\Roaming\Ultralytics\settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "C:\\Dokumen\\Reproduce Jurnal\\datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "ae3ec6e16f9143d28ba9ce8bce4b92432183c9802c009a79ebdd863686a8f350",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}


In [4]:
best_weight = os.path.join(PROJECT_DIR, "baseline_yolov8n", "weights", "best.pt")
assert os.path.isfile(best_weight), f"Best weight not found: {best_weight}"

best_model = YOLO(best_weight)

split_configs = {
    "Validation": None,
    "Test":       DATA_YAML,
    "Train":      None,
}

print("=" * 70)
print("EVALUASI MODEL BASELINE YOLOv8n PADA SETIAP SPLIT")
print("=" * 70)

all_metrics = {}

# --- Validation ---
print("\n[1/3] Evaluasi pada Data Validation ...")
val_metrics = best_model.val(
    data=DATA_YAML,
    split="val",
    project=PROJECT_DIR,
    name="eval_validation",
    exist_ok=True,
)
all_metrics["Validation"] = val_metrics

# --- Test ---
print("\n[2/3] Evaluasi pada Data Test ...")
test_metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    project=PROJECT_DIR,
    name="eval_test",
    exist_ok=True,
)
all_metrics["Test"] = test_metrics

# --- Train ---
print("\n[3/3] Evaluasi pada Data Train ...")
train_yaml_content = {
    "path": SPLIT_ROOT,
    "train": os.path.join("Data Train", "images"),
    "val":   os.path.join("Data Train", "images"),
    "test":  os.path.join("Data Train", "images"),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
TRAIN_EVAL_YAML = os.path.join(SPLIT_ROOT, "baseline_eval_train.yaml")
with open(TRAIN_EVAL_YAML, "w") as f:
    yaml.dump(train_yaml_content, f, default_flow_style=False)

train_metrics = best_model.val(
    data=TRAIN_EVAL_YAML,
    split="val",
    project=PROJECT_DIR,
    name="eval_train",
    exist_ok=True,
)
all_metrics["Train"] = train_metrics

print("\nEvaluasi selesai untuk semua split.")

EVALUASI MODEL BASELINE YOLOv8n PADA SETIAP SPLIT

[1/3] Evaluasi pada Data Validation ...
Ultralytics 8.4.40  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 860.288.0 MB/s, size: 1506.0 KB)
val: Scanning C:\Dokumen\SKRIPSI\Bimbingan\SPLIT DATA ROBUST\SPLIT DATA REMAP 9 KELAS\Data Validation\labels.cache... 56 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 56/56  0.0s


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.4s/it 17.4s6.9ss
                   all         56       6187      0.228      0.188      0.128     0.0422
         atrophic_scar         55       1758      0.251       0.15     0.0844     0.0215
                comedo         56       1787     0.0879    0.00951    0.00521    0.00204
     hypertrophic_scar         36        142      0.124     0.0282    0.00816    0.00191
               melasma         46        728       0.28      0.309        0.2      0.051
                 nevus         55        297      0.455      0.441      0.351      0.123
                nodule         19         34      0.202      0.324      0.194     0.0853
                 other         34         81          0          0          0          0
                papule         56       1129      0.373      0.347      0.225     0.0656
               pustule         45        231      0.277     0.0823  

"C:\Dokumen\SKRIPSI\Jurnal\Delima\Delima"

In [5]:
rows = []
for split_name, m in all_metrics.items():
    box = m.box
    rows.append({
        "Split": split_name,
        "mAP50": round(box.map50, 4),
        "mAP50-95": round(box.map, 4),
        "Precision": round(box.mp, 4),
        "Recall": round(box.mr, 4),
    })

df_summary = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("RINGKASAN PERFORMA BASELINE YOLOv8n")
print("=" * 70)
print(df_summary.to_string(index=False))
print("=" * 70)

# Per-class breakdown for each split
for split_name, m in all_metrics.items():
    box = m.box
    per_class = []
    for i, cname in enumerate(CLASS_NAMES):
        per_class.append({
            "Class": cname,
            "Precision": round(box.p[i], 4) if i < len(box.p) else None,
            "Recall": round(box.r[i], 4) if i < len(box.r) else None,
            "mAP50": round(box.ap50()[i], 4) if i < len(box.ap50()) else None,
            "mAP50-95": round(box.ap[i], 4) if i < len(box.ap) else None,
        })
    df_cls = pd.DataFrame(per_class)
    print(f"\n--- Per-Class Metrics: {split_name} ---")
    print(df_cls.to_string(index=False))


RINGKASAN PERFORMA BASELINE YOLOv8n
     Split  mAP50  mAP50-95  Precision  Recall
Validation 0.1280    0.0422     0.2278  0.1878
      Test 0.1241    0.0371     0.2251  0.1959
     Train 0.2290    0.0888     0.3690  0.2782


TypeError: 'numpy.ndarray' object is not callable

In [ ]:
results_csv = os.path.join(PROJECT_DIR, "baseline_yolov8n", "results.csv")
if os.path.isfile(results_csv):
    df_train_log = pd.read_csv(results_csv)
    df_train_log.columns = df_train_log.columns.str.strip()
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(df_train_log["epoch"], df_train_log["metrics/mAP50(B)"], label="mAP50")
    axes[0, 0].plot(df_train_log["epoch"], df_train_log["metrics/mAP50-95(B)"], label="mAP50-95")
    axes[0, 0].set_title("mAP per Epoch")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("mAP")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    axes[0, 1].plot(df_train_log["epoch"], df_train_log["metrics/precision(B)"], label="Precision")
    axes[0, 1].plot(df_train_log["epoch"], df_train_log["metrics/recall(B)"], label="Recall")
    axes[0, 1].set_title("Precision & Recall per Epoch")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    axes[1, 0].plot(df_train_log["epoch"], df_train_log["train/box_loss"], label="Train Box Loss")
    axes[1, 0].plot(df_train_log["epoch"], df_train_log["val/box_loss"], label="Val Box Loss")
    axes[1, 0].set_title("Box Loss per Epoch")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Loss")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    axes[1, 1].plot(df_train_log["epoch"], df_train_log["train/cls_loss"], label="Train Cls Loss")
    axes[1, 1].plot(df_train_log["epoch"], df_train_log["val/cls_loss"], label="Val Cls Loss")
    axes[1, 1].set_title("Classification Loss per Epoch")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.suptitle("Training Curves - Baseline YOLOv8n", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\nTraining stopped at epoch {df_train_log['epoch'].max()} (early stopping patience=50)")
else:
    print("results.csv not found – run training first.")

<Figure size 1400x1000 with 4 Axes>


Training stopped at epoch 204 (early stopping patience=50)
